# V8 · Current public leader + verified coefficient refinement

**Previous confirmed Public Score: 22.99915 (our V6).** This is a new version
of the same notebook; its history and URL are preserved. V7 is also submitted
and its official result was still pending when this update was prepared.

On 30 October we selected [heliosli V26](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy?scriptVersionId=363902440),
the public notebook leader at **23.00037**, copied its pinned 18-file portfolio,
and continued the chemistry optimization. The new Public Score is **pending**.
The measured local score and a clearly labelled public projection appear below.

## What changed since V6

| Component | Our V6 | V8 baseline and refinement |
|---|---|---|
| Inverse square root | 62 qubits | same verified yoyobar / Özer circuit |
| DQI | hengck23 iteration 014 | identical bytes; full original error/repetition penalty |
| Chemistry loaders | 15-slot portfolio | identical compiled loaders |
| Chemistry coefficients | effective normalization 48.33104410 | leader 48.32336162 → selected checkpoint **48.3229686033** |

The improved starting point comes from the credited new public weights. **Our additional
gain is small:** constrained offset/rank fitting and exact pair polishing of
the chemistry coefficients. We selected the constrained rank/offset candidate and exact pair polish
by the native accuracy checks and normalization.
Only `dfthcsos/submission.npz` may differ from the new public leader.

## Verification and reproduction

Use **CPU**, internet off, then **Run All**. The attached input is frozen at
heliosli **V26**, and all 18 member hashes are authenticated before extraction.
The run compares the leader and V8 with two fresh **78,080-shot** native
chemistry checks, proves the ANGLE behavior for every reachable address and
measurement outcome, and writes `submission.zip` and `verification_receipt_v8.json`.

The unchanged inverse has our fresh complete **200,540,928-shot** official
receipt. The unchanged DQI has our complete **102,400-shot** receipt; both
are tied to the exact circuit SHA-256. Set `RUN_FULL_UNCHANGED_VERIFIERS=True`
to rerun both longer checks. All verifiers and their scoring penalties remain
unchanged. Production normally takes a few minutes; rerunning all unchanged
circuits takes longer.

The DQI decoder is approximate and omits some rare low-weight guards. A sampled
PASS does not prove exactness. Local and public scores can differ because the
organizers use different salts. No salt or leaderboard feedback selects candidates.

Search sources, seeds, successful and rejected candidate records, the selected
float64 checkpoint, the original V26 notebook and its credits are included.
The long data cells are folded for readability.


In [ ]:
import os,sys,io,json,math,hashlib,shutil,subprocess,time,zipfile,base64,zlib
from pathlib import Path
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
import numpy as np
from IPython.display import display,Markdown
INPUT=Path(os.environ.get('QCC_INPUT','/kaggle/input'))
WORK=Path(os.environ.get('QCC_WORK','/kaggle/working'));WORK.mkdir(parents=True,exist_ok=True)
SLUG='google-verifiable-quantum-circuit-compilation'
COMP=next(p for p in [Path(os.environ.get('QCC_COMP_DIR','/nonexistent')),INPUT/'competitions'/SLUG,INPUT/SLUG] if (p/'verifiers').is_dir())
SUB=WORK/'submission_v8';SUB.mkdir(exist_ok=True)
SRC=WORK/'source_v8';SRC.mkdir(exist_ok=True)
native=[os.environ['QCC_NATIVE_DIR']] if os.environ.get('QCC_NATIVE_DIR') else [str(COMP/'kickmix')]
sys.path[:0]=native+[str(COMP/'verifiers'),str(SRC)]
import kickmix as km
from dfthcsos import verify as chemistry
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
started=time.monotonic()


## Authenticate heliosli V26 and measure the baseline

In [ ]:
PINS={'dfthcsos/angle_k_0.kmb': '9c31da0c1d047e7dcac638df807d3cef96eac6f736a8838a0ae6943e9c46c74f', 'dfthcsos/angle_k_1.kmb': '39b8d10098be097f75148639ee302249bdabe112a9278a48565eadb0cb86810b', 'dfthcsos/angle_k_10.kmb': 'e75f6b0685d754bf76f3f4d62589bf1d07b118faeeb0ab8745833733286db2fc', 'dfthcsos/angle_k_11.kmb': '773e54f6f0d7c3f769db433f263d541cdcff1076fb75109740907e3249996141', 'dfthcsos/angle_k_12.kmb': 'd86ca7ddd984059ded765b953cc936388517a7ca5789676ae90910a2be89769d', 'dfthcsos/angle_k_13.kmb': '50d82cf65116f06c0fe469ebba312f72675b277263461ab88b4d15eca947c050', 'dfthcsos/angle_k_14.kmb': 'ae2070998e31b7e05d01183914df20b6974c1760bddf90d1adac86ea75c7bf3f', 'dfthcsos/angle_k_2.kmb': '2e8dfdeb69c496e9e510fe99722153663760980d45c35ad9b72e79ba799dde49', 'dfthcsos/angle_k_3.kmb': '13817cf8d36ea112e62d7863edde2193aa5907dd0605612c4f0e1e54bacc173b', 'dfthcsos/angle_k_4.kmb': '8dce8f4a380998d818bf8a08e4f3daa62aaeb31a25d084aa57549f931a937f3d', 'dfthcsos/angle_k_5.kmb': '42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099', 'dfthcsos/angle_k_6.kmb': '7fd7bd333b1427ecf546b0c27717ed9428e255e5165c4db14e5f95f8af45c838', 'dfthcsos/angle_k_7.kmb': '809a775b98836ee00e8783d3f616bb8517bb3a3621978bc704d20b543a8df4ec', 'dfthcsos/angle_k_8.kmb': '42d0deb376981d0470bebdd370ca2adf79685b4bf3e6909be262e2beced030ec', 'dfthcsos/angle_k_9.kmb': 'f1c49e7cf8e4f3a9adaffe167d2bbe21497a52eca2c7d09f765fc5a125e304c3', 'dfthcsos/submission.npz': '0f1426b039356378126e2d1a1bfbfb7e0c66d5b45226ca84942200ea79a97f45', 'dqi.kmb': '7a378f4755605e0c5fa6117f808891bf8dcf0797d047155e6d4c9e75e3c99421', 'invsqrt.kmb': '6dbb98b34c3b16f1c4e019350c10d42259a6faa97e464c9b7c9390f877f392f4'}
UPSTREAM_ZIP_SHA256='ca7e32eb0daa2ce647d9a0d0d5dc76488daa38a8897dbf1074153efc3aaa337b'
archive=next(p for p in INPUT.rglob('submission.zip') if sha(p)==UPSTREAM_ZIP_SHA256)
with zipfile.ZipFile(archive) as z:
    assert len(z.namelist())==18 and set(z.namelist())==set(PINS)
    for name,pin in PINS.items():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
        path=SUB/name;path.parent.mkdir(parents=True,exist_ok=True)
        with z.open(name) as fi,path.open('wb') as fo:shutil.copyfileobj(fi,fo)
        assert sha(path)==pin,name
baseline_fresh=chemistry.verify(str(SUB/'dfthcsos'),salt='notebook-verification')
assert baseline_fresh['status']=='PASS' and baseline_fresh['verified_shots']==78080
(WORK/'baseline_chemistry_v8.json').write_text(json.dumps(baseline_fresh,indent=2))
print('All 18 files authenticated. Baseline chemistry cost:',baseline_fresh['score'])


In [ ]:
SOURCES={'gauge_search.py': '"""Exact pairwise orthogonal descent of augmented DFTHC coefficient columns.\n\nRetains A A.T for A=[W; H2_I]. The piecewise trigonometric L1 objective\nis minimized over all zero crossings and all interior stationary points.\n"""\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\n\n\ndef objective(a):\n    return float(0.25 * np.square(np.abs(a).sum(axis=-2)).sum())\n\n\ndef pair_minimum(x, y, radius):\n    # Roots of x*cos(t)+y*sin(t) and y*cos(t)-x*sin(t), modulo pi.\n    roots = np.concatenate([np.arctan2(-x, y), np.arctan2(y, x)])\n    roots = (roots + np.pi / 2) % np.pi - np.pi / 2\n    cuts = np.unique(np.r_[-radius, 0., roots[(roots > -radius) & (roots < radius)], radius])\n    mids = (cuts[1:] + cuts[:-1]) / 2\n    c, s = np.cos(mids)[:, None], np.sin(mids)[:, None]\n    sx, sy = np.sign(c*x+s*y), np.sign(c*y-s*x)\n    p, q = sx@x, sx@y\n    r, u = sy@y, -sy@x\n    aa, bb, cc = p*p+r*r, q*q+u*u, p*q+r*u\n    base = .5*np.arctan2(-2*cc, bb-aa)\n    inside = []\n    for shift in (-np.pi/2, 0., np.pi/2):\n        points = base+shift\n        inside.extend(points[(points > cuts[:-1]) & (points < cuts[1:])])\n    theta = np.r_[cuts, inside]\n    c, s = np.cos(theta)[:, None], np.sin(theta)[:, None]\n    values = np.square(np.abs(c*x+s*y).sum(1)) + np.square(np.abs(c*y-s*x).sum(1))\n    idx = int(np.argmin(values))\n    return float(theta[idx]), float(values[idx])\n\n\ndef search(source, output, sweeps=5, radius=.03, seed=20261029):\n    with np.load(source, allow_pickle=False) as z:\n        data = {k: np.array(z[k]) for k in z.files}\n    original = np.concatenate([data[\'W\'], data[\'H2_I\'][:, None]], axis=1)\n    a = original.copy()\n    rotation = np.tile(np.eye(a.shape[-1]), (len(a), 1, 1))\n    before = objective(a)\n    rng = np.random.default_rng(seed)\n    history = []\n    started = time.time()\n    pairs = [(r,i,j) for r in range(len(a)) for i in range(a.shape[-1]) for j in range(i+1,a.shape[-1])]\n    for sweep in range(sweeps):\n        accepted = 0\n        for index in rng.permutation(len(pairs)):\n            r, i, j = pairs[index]\n            x,y = a[r,:,i].copy(),a[r,:,j].copy()\n            old = np.abs(x).sum()**2+np.abs(y).sum()**2\n            angle, value = pair_minimum(x,y,radius)\n            if value < old-1e-13:\n                c,s=np.cos(angle),np.sin(angle)\n                a[r,:,i],a[r,:,j]=c*x+s*y,c*y-s*x\n                ri,rj=rotation[r,:,i].copy(),rotation[r,:,j].copy()\n                rotation[r,:,i],rotation[r,:,j]=c*ri+s*rj,c*rj-s*ri\n                accepted+=1\n        record=dict(sweep=sweep,objective=objective(a),accepted=accepted,seconds=time.time()-started)\n        history.append(record)\n        print(json.dumps(record),flush=True)\n        data[\'W\'],data[\'H2_I\']=a[:,:-1],a[:,-1]\n        if \'offset_W\' in data:data[\'offset_W\']=data[\'H2_I\']-data[\'W\'].sum(1)\n        output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)\n        np.savez(output,**data)\n        np.save(output.with_suffix(\'.rotation.npy\'),rotation)\n        defect=float(np.max(np.abs(a@a.transpose(0,2,1)-original@original.transpose(0,2,1))))\n        report=dict(before=before,after=objective(a),gram_defect=defect,seed=seed,radius=radius,history=history)\n        output.with_suffix(\'.json\').write_text(json.dumps(report,indent=2))\n        assert defect < 1e-9\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'source\');p.add_argument(\'output\')\n    p.add_argument(\'--sweeps\',type=int,default=5);p.add_argument(\'--radius\',type=float,default=.03)\n    a=p.parse_args();search(a.source,a.output,a.sweeps,a.radius)\n', 'gauge_cayley_search.py': '"""CPU float64 multicolumn Cayley search, followed by exact-objective selection."""\nimport argparse,json,time\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.optimize import minimize\nfrom gauge_search import objective\n\ndef run(source,output,iterations=200,carry=False,perturb=0.,seed=0):\n    z=dict(np.load(source,allow_pickle=False));A=np.concatenate([z[\'W\'],z[\'H2_I\'][:,None]],axis=1)\n    r,b,c=A.shape;eye=np.tile(np.eye(c),(r,1,1));ii,jj=np.triu_indices(c,1)\n    best=A.copy();current=A.copy();initial=objective(A);bestval=initial;history=[];start=time.time()\n    if perturb:\n        rng=np.random.default_rng(seed);X=rng.normal(size=(r,c,c))*perturb/np.sqrt(c);X=X-X.transpose(0,2,1)\n        current=A@np.linalg.solve(eye-X,eye+X)\n    for eps in (1e-3,1e-4,1e-5,1e-6,1e-7,1e-8):\n        base=(current if carry else best).copy();nparams=r*len(ii)\n        def fg(flat,check=False):\n            K=np.zeros((r,c,c));K[:,ii,jj]=flat.reshape(r,-1);X=K-K.transpose(0,2,1)\n            M=eye-X;R=np.linalg.solve(M,eye+X);B=base@R\n            smooth=np.sqrt(B*B+eps*eps);l1=smooth.sum(1)\n            f=.25*np.square(l1).sum();G=.5*l1[:,None,:]*B/smooth\n            H=base.transpose(0,2,1)@G\n            D=np.linalg.solve(M.transpose(0,2,1),H)@(eye+R).transpose(0,2,1)\n            grad=(D-D.transpose(0,2,1))[:,ii,jj].ravel()\n            return (f,grad,B) if check else (f,grad)\n        if not history:\n            q=np.zeros(nparams);direction=np.random.default_rng(1793).normal(size=nparams);direction/=np.linalg.norm(direction)\n            f,g=fg(q);h=1e-6;num=(fg(q+h*direction)[0]-fg(q-h*direction)[0])/(2*h)\n            assert abs(num-g@direction)<1e-5,(num,g@direction)\n            print(\'Gradient directional check:\',num,float(g@direction),flush=True)\n        result=minimize(fg,np.zeros(nparams),method=\'L-BFGS-B\',jac=True,options=dict(maxiter=iterations,ftol=1e-14,gtol=1e-8,maxls=30,maxcor=10))\n        f,g,B=fg(result.x,True);value=objective(B)\n        if value<bestval:bestval=value;best=B.copy()\n        current=B.copy()\n        record=dict(eps=eps,iterations=int(result.nit),exact=value,best=bestval,seconds=time.time()-start)\n        history.append(record);print(json.dumps(record),flush=True)\n        z[\'W\'],z[\'H2_I\']=best[:,:-1],best[:,-1]\n        if \'offset_W\' in z:z[\'offset_W\']=z[\'H2_I\']-z[\'W\'].sum(1)\n        output=Path(output);np.savez(output,**z)\n        defect=float(np.max(np.abs(best@best.transpose(0,2,1)-A@A.transpose(0,2,1))))\n        assert defect<1e-10\n        output.with_suffix(\'.json\').write_text(json.dumps(dict(before=initial,after=bestval,gram_defect=defect,carry=carry,perturb=perturb,seed=seed,history=history),indent=2))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'source\');p.add_argument(\'output\');p.add_argument(\'--iterations\',type=int,default=200)\n    p.add_argument(\'--carry\',action=\'store_true\');p.add_argument(\'--perturb\',type=float,default=0.);p.add_argument(\'--seed\',type=int,default=0)\n    a=p.parse_args();run(a.source,a.output,a.iterations,a.carry,a.perturb,a.seed)\n', 'gauge_multistart.py': '"""Two-worker continuation search over exact augmented orthogonal gauges.\n\nSelection uses the deterministic L1 objective, never a simulator seed or a\nleaderboard response. A stop file prevents new trials after the goal ends.\n"""\nimport argparse,json,os,shutil,subprocess,sys,time\nfrom concurrent.futures import ThreadPoolExecutor,as_completed\nfrom pathlib import Path\nimport numpy as np\nfrom gauge_search import objective\n\np=argparse.ArgumentParser();p.add_argument(\'source\',type=Path);p.add_argument(\'out\',type=Path)\np.add_argument(\'--rounds\',type=int,default=3);p.add_argument(\'--trials\',type=int,default=6)\na=p.parse_args();a.out.mkdir(parents=True,exist_ok=True)\nstop=a.out/\'STOP\';history=[];source=a.source.resolve();started=time.time()\ndef value(path):\n    with np.load(path,allow_pickle=False) as z:return objective(np.concatenate([z[\'W\'],z[\'H2_I\'][:,None]],axis=1))\nbestval=value(source)\nfor generation in range(a.rounds):\n    if stop.exists():break\n    initial=a.out/f\'initial_{generation}.npz\';shutil.copyfile(source,initial)\n    def trial(index):\n        if stop.exists():return dict(skipped=True)\n        seed=20261029+generation*101+index*37;perturb=(.04,.07,.1,.12,.15,.18)[index%6]\n        out=a.out/f\'g{generation}_t{index}.npz\'\n        cmd=[sys.executable,\'-u\',str(Path(__file__).with_name(\'gauge_cayley_search.py\')),str(initial),str(out),\'--iterations\',\'5000\',\'--carry\',\'--perturb\',str(perturb),\'--seed\',str(seed)]\n        with out.with_suffix(\'.log\').open(\'w\') as f:subprocess.run(cmd,env=os.environ,stdout=f,stderr=subprocess.STDOUT,check=True)\n        r=json.loads(out.with_suffix(\'.json\').read_text());return dict(path=str(out),seed=seed,perturb=perturb,after=r[\'after\'],gram_defect=r[\'gram_defect\'])\n    results=[]\n    with ThreadPoolExecutor(max_workers=2) as pool:\n        for future in as_completed([pool.submit(trial,i)for i in range(a.trials)]):\n            r=future.result()\n            if r.get(\'skipped\'):continue\n            assert r[\'gram_defect\']<1e-10;results.append(r)\n            print(json.dumps(dict(generation=generation,**r)),flush=True)\n    if results:\n        selected=min(results,key=lambda r:r[\'after\'])\n        if selected[\'after\']<bestval:source=Path(selected[\'path\']).resolve();bestval=selected[\'after\']\n    history.append(dict(generation=generation,results=results,best_path=str(source),best_objective=bestval,seconds=time.time()-started))\n    (a.out/\'history.json\').write_text(json.dumps(history,indent=2))\n    (a.out/\'best_path.txt\').write_text(str(source))\nprint(\'BEST\',source,bestval,flush=True)\n', 'research_cpu/fixed_word_joint64_v6.py': '# V6 change: 19.99995 mHa feasibility cap, retaining 0.00005 mHa guard.\n# CPU adaptation of Ahmed Berat Ozer\'s v11 fixed_word_joint64.py (Apache-2.0).\n# Device changed to CPU; feasibility caps leave numerical safety margins.\n"""Full float64 constrained W/identity fit with fixed XOR-table bytes.\n\nNew control: the published Kirill joint study used float32 MPS and lost gap\naccuracy near 22,000 Ha. Here the true one-body, two-body and gap constraints\nare differentiated in float64, with QR-whitened physical coordinates and an\nanchored two-body residual. This is not another exact column gauge search.\n"""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport numpy as np\nfrom scipy.optimize import minimize\nfrom scipy.optimize import linear_sum_assignment\nimport torch\nCOMP=Path(os.environ[\'QCC_COMP_DIR\']);sys.path[:0]=[str(COMP/\'verifiers\')]\nfrom dfthcsos import verify as V\nfrom dfthcsos.scoreinfra import dfthc_math as M\nimport q8lib as L\nap=argparse.ArgumentParser();ap.add_argument(\'initial\',type=Path);ap.add_argument(\'out\',type=Path)\nap.add_argument(\'--evaluations\',type=int,default=650)\nap.add_argument(\'--fixed-gap\',action=\'store_true\',help=\'Implicitly eliminate the stiff global offset scale\')\nap.add_argument(\'--iterations\',type=int,default=180)\nap.add_argument(\'--smooth-l1\',type=float,default=0.)\na=ap.parse_args();a.out.mkdir(parents=True,exist_ok=True)\ntorch.set_num_threads(1);torch.set_default_dtype(torch.float64)\n# CPU float64; no CUDA backend\nstart=time.time();z=dict(np.load(a.initial,allow_pickle=False));inp=V.load_inputs(a.initial)\nbase=V.evaluate_words(inp,z[\'angle_words\'],z[\'tree_edges\'],int(z[\'brot\']))\nhalf=2**(int(z[\'brot\'])-1);signed=((z[\'angle_words\'].astype(np.int64)+half)%(2*half))-half\nall_u=M.reconstruct_u_vectors(signed/half,z[\'tree_edges\']);all_u/=np.linalg.norm(all_u,axis=-1,keepdims=True)\nuo,u=M.split_u_vectors(all_u,z[\'W\']);r,b,c=z[\'W\'].shape\ntarget=L.TwoBody(\'cpu\',torch.float64)\nut=torch.tensor(u,device=\'cpu\');projectors=target.pair(ut).transpose(1,2)\nQ,Rt=torch.linalg.qr(projectors)\nw0=torch.tensor(z[\'W\'],device=\'cpu\');v0=Rt@w0\nK0=v0@v0.transpose(1,2)\ncross=torch.einsum(\'rpi,spj->rsij\',Q,Q)\nprojected_target=Q.transpose(1,2)@target.T@Q\nE0=torch.einsum(\'rsij,sjk,rslk->ril\',cross,K0,cross)-projected_target\nx0=torch.einsum(\'rpi,ric->rcp\',Q,v0).reshape(r*c,-1)\nerror0=(x0.T@x0-target.T).square().sum()\noffset0=torch.tensor(inp.W,device=\'cpu\')\nh1=inp.ref[\'oei\']-.5*np.einsum(\'prrq->pq\',inp.ref[\'tei\'])\nconstant=torch.tensor(h1-.5*inp.ref[\'electrons\']*inp.S+inp.beta_1*np.eye(76),device=\'cpu\')\nenergy=float(inp.energy_upper-inp.ref[\'ecore\']+inp.ref[\'electrons\']*inp.beta_1)\none=torch.tensor(uo,device=\'cpu\')\nhp0=constant-torch.einsum(\'rb,rbp,rbq->pq\',(w0*offset0[:,None]).sum(-1),ut,ut)\n_,eigvec=torch.linalg.eigh(hp0)\nrows,cols=linear_sum_assignment(-np.abs(uo@eigvec.cpu().numpy()))\npermutation=np.empty(76,dtype=np.int64);permutation[rows]=cols\nperm=torch.tensor(permutation,device=\'cpu\')\ncut=r*b*c;size=cut+r*c;calls=0;best=float(base[\'lambda_eff\']);best_z=np.zeros(size);history=[]\ne_cap=.01999995;gap_floor=3.9149\n\ndef quantities(flat):\n    dv=flat[:cut].reshape(r,b,c);off=offset0+flat[cut:].reshape(r,c)\n    vv=v0+dv;w=torch.linalg.solve_triangular(Rt,vv,upper=True)\n    delta=v0@dv.transpose(1,2)+dv@v0.transpose(1,2)+dv@dv.transpose(1,2)\n    dm=torch.einsum(\'rsij,sjk,rslk->ril\',cross,delta,cross)\n    e2sq=error0+2*(E0*delta).sum()+(delta*dm).sum()\n    if a.fixed_gap:\n        # Solve the scalar physical gap equation outside autograd, then attach\n        # its exact implicit first derivative. No eigenvector or shift changes.\n        C=torch.einsum(\'rb,rbp,rbq->pq\',(w*off[:,None]).sum(-1),ut,ut)\n        with torch.no_grad():\n            alpha=torch.ones((),device=\'cpu\')\n            offnorm=off.square().sum()\n            for _ in range(8):\n                ee,qq=torch.linalg.eigh(constant-alpha*C)\n                f=energy+ee.abs().sum()-ee.sum()+.5*alpha.square()*offnorm-gap_floor\n                projected=torch.diagonal(qq.T@C@qq)\n                derivative=alpha*offnorm-((ee.sign()-1)*projected).sum()\n                alpha-=f/derivative\n                if abs(float(f))<2e-12:break\n        ee=torch.linalg.eigvalsh(constant-alpha*C)\n        f=energy+ee.abs().sum()-ee.sum()+.5*alpha.square()*off.square().sum()-gap_floor\n        implicit_alpha=alpha-f/derivative\n        off=off*implicit_alpha\n    hp=constant-torch.einsum(\'rb,rbp,rbq->pq\',(w*off[:,None]).sum(-1),ut,ut)\n    ev=torch.linalg.eigvalsh(hp)\n    hq=torch.einsum(\'j,jp,jq->pq\',ev[perm],one,one)\n    e1sq=(hq-hp).square().sum()\n    lam1=ev.abs().sum()\n    lam=lam1+.25*(w.abs().sum(1)+(off+w.sum(1)).abs()).square().sum()\n    gap=energy+lam1-ev.sum()+.5*off.square().sum()\n    effective=(gap*(2*lam-gap)).sqrt()\n    return effective,e1sq,e2sq,gap,lam,w,off\n\nzero=torch.zeros(size,device=\'cpu\');initial=quantities(zero)\ninitial_values=[float(initial[i]) for i in range(5)]\nif not a.fixed_gap:\n    assert abs(initial_values[0]-base[\'lambda_eff\'])<1e-7\n    assert abs(np.sqrt(initial_values[1])-base[\'error_1\'])<1e-8\nassert abs(np.sqrt(initial_values[2])-base[\'error_2\'])<1e-8\nprint(json.dumps({\'event\':\'native_fixture_PASS\',\'initial\':initial_values,\'native\':{k:float(base[k]) for k in (\'lambda_eff\',\'error_1\',\'error_2\',\'E_gap\')},\'seconds\':time.time()-start}),flush=True)\n\nclass Limit(Exception):pass\ndef objective(flat_np,mu):\n    global calls,best,best_z\n    calls+=1\n    if calls>a.evaluations:raise Limit\n    flat=torch.tensor(flat_np,device=\'cpu\',requires_grad=True)\n    lam,e1,e2,gap,Lam,_,_=quantities(flat)\n    constraints=torch.stack([1-e1/e_cap**2,1-e2/e_cap**2] if a.fixed_gap else\n                            [1-e1/e_cap**2,1-e2/e_cap**2,gap-gap_floor])\n    valid=bool((constraints>0).all()) and (not a.fixed_gap or abs(float(gap.detach())-gap_floor)<2e-8)\n    if valid:\n        surrogate=lam\n        if a.smooth_l1:\n            ww,hh=quantities(flat)[-2:]\n            eps=a.smooth_l1\n            smooth=.25*(torch.sqrt(ww.square()+eps*eps).sum(1)+torch.sqrt((hh+ww.sum(1)).square()+eps*eps)).square().sum()\n            exact=.25*(ww.abs().sum(1)+(hh+ww.sum(1)).abs()).square().sum()\n            surrogate=(gap*(2*(Lam+smooth-exact)-gap)).sqrt()\n        loss=surrogate.log()-mu*constraints.log().sum()\n        if float(lam.detach())<best:\n            best=float(lam.detach());best_z=flat_np.copy()\n    else:\n        loss=1000+1e5*torch.minimum(constraints,torch.zeros_like(constraints)).square().sum()\n    loss.backward();grad=flat.grad.detach().cpu().numpy()\n    if calls%25==0:\n        rec={\'call\':calls,\'lambda_eff\':float(lam.detach()),\'best\':best,\'error_1\':float(e1.detach().sqrt()),\n             \'error_2\':float(e2.detach().sqrt()),\'gap\':float(gap.detach()),\'mu\':mu,\'seconds\':time.time()-start}\n        history.append(rec);print(json.dumps(rec),flush=True)\n        np.save(a.out/\'best_coordinates.npy\',best_z)\n        (a.out/\'history.json\').write_text(json.dumps(history,indent=2))\n    return float(loss.detach()),grad\n\nif a.fixed_gap:\n    torch.manual_seed(20261029)\n    probe=torch.randn(size,device=\'cpu\');probe/=probe.norm()\n    # Check away from the exact L1 kinks of a sparse optimized gauge.\n    center=1e-3*probe\n    point=center.detach().clone().requires_grad_(True)\n    def smooth_probe(x):\n        v=quantities(x)\n        return v[1]+v[2]+v[3]+1e-4*(torch.sqrt(v[-2].square()+1e-8).sum()+torch.sqrt(v[-1].square()+1e-8).sum())\n    val=smooth_probe(point);val.backward();analytic=float(point.grad@probe)\n    with torch.no_grad():fd=float((smooth_probe(center+1e-6*probe)-smooth_probe(center-1e-6*probe))/2e-6)\n    assert abs(fd-analytic)<1e-5*max(1.,abs(analytic)),(fd,analytic)\n    print(json.dumps({\'implicit_gap_gradient\':\'PASS\',\'analytic\':analytic,\'finite_difference\':fd}),flush=True)\ncurrent=np.zeros(size)\ntry:\n    for mu in (1e-5,1e-6,1e-7):\n        opt=minimize(lambda x:objective(x,mu),current,jac=True,method=\'L-BFGS-B\',\n                     options={\'maxiter\':a.iterations,\'maxcor\':30,\'ftol\':1e-15 if a.fixed_gap else 1e-13,\'gtol\':1e-9,\'maxls\':30})\n        current=opt.x.copy()\nexcept Limit:print(\'evaluation budget reached\',flush=True)\nwith torch.no_grad():\n    got=quantities(torch.tensor(best_z,device=\'cpu\'));w,off=got[-2:]\n    result=dict(z);result[\'W\']=w.cpu().numpy();result[\'H2_I\']=(off+w.sum(1)).cpu().numpy();result[\'offset_W\']=off.cpu().numpy()\nnp.savez(a.out/\'submission.npz\',**result)\nnative=V.evaluate_words(V.load_inputs(a.out/\'submission.npz\'),result[\'angle_words\'],result[\'tree_edges\'],int(result[\'brot\']))\nreport={\'status\':\'numerical_candidate\',\'before\':{k:float(base[k]) for k in (\'lambda_eff\',\'error_1\',\'error_2\',\'E_gap\')},\n        \'after\':{k:float(native[k]) for k in (\'lambda_eff\',\'error_1\',\'error_2\',\'E_gap\')},\'calls\':calls,\'seconds\':time.time()-start,\n        \'words_identical\':bool(np.array_equal(result[\'angle_words\'],z[\'angle_words\']))}\n(a.out/\'report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)\n', 'research_cpu/rank_offset_sqp_v6.py': '# V6 change: 19.99995 mHa feasibility cap, retaining 0.00005 mHa guard.\n# CPU adaptation of Ahmed Berat Ozer\'s v11 fixed_word_joint64.py (Apache-2.0).\n# Device changed to CPU; feasibility caps leave numerical safety margins.\n"""Full float64 constrained W/identity fit with fixed XOR-table bytes.\n\nNew control: the published Kirill joint study used float32 MPS and lost gap\naccuracy near 22,000 Ha. Here the true one-body, two-body and gap constraints\nare differentiated in float64, with QR-whitened physical coordinates and an\nanchored two-body residual. This is not another exact column gauge search.\n"""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport numpy as np\nfrom scipy.optimize import minimize\nfrom scipy.optimize import linear_sum_assignment\nimport torch\nCOMP=Path(os.environ[\'QCC_COMP_DIR\']);sys.path[:0]=[str(COMP/\'verifiers\')]\nfrom dfthcsos import verify as V\nfrom dfthcsos.scoreinfra import dfthc_math as M\nimport q8lib as L\nap=argparse.ArgumentParser();ap.add_argument(\'initial\',type=Path);ap.add_argument(\'out\',type=Path)\nap.add_argument(\'--evaluations\',type=int,default=650)\nap.add_argument(\'--fixed-gap\',action=\'store_true\',help=\'Implicitly eliminate the stiff global offset scale\')\nap.add_argument(\'--iterations\',type=int,default=180)\nap.add_argument(\'--smooth-l1\',type=float,default=0.)\na=ap.parse_args();a.out.mkdir(parents=True,exist_ok=True)\ntorch.set_num_threads(1);torch.set_default_dtype(torch.float64)\n# CPU float64; no CUDA backend\nstart=time.time();z=dict(np.load(a.initial,allow_pickle=False));inp=V.load_inputs(a.initial)\nbase=V.evaluate_words(inp,z[\'angle_words\'],z[\'tree_edges\'],int(z[\'brot\']))\nhalf=2**(int(z[\'brot\'])-1);signed=((z[\'angle_words\'].astype(np.int64)+half)%(2*half))-half\nall_u=M.reconstruct_u_vectors(signed/half,z[\'tree_edges\']);all_u/=np.linalg.norm(all_u,axis=-1,keepdims=True)\nuo,u=M.split_u_vectors(all_u,z[\'W\']);r,b,c=z[\'W\'].shape\ntarget=L.TwoBody(\'cpu\',torch.float64)\nut=torch.tensor(u,device=\'cpu\');projectors=target.pair(ut).transpose(1,2)\nQ,Rt=torch.linalg.qr(projectors)\nw0=torch.tensor(z[\'W\'],device=\'cpu\');v0=Rt@w0\nK0=v0@v0.transpose(1,2)\ncross=torch.einsum(\'rpi,spj->rsij\',Q,Q)\nprojected_target=Q.transpose(1,2)@target.T@Q\nE0=torch.einsum(\'rsij,sjk,rslk->ril\',cross,K0,cross)-projected_target\nx0=torch.einsum(\'rpi,ric->rcp\',Q,v0).reshape(r*c,-1)\nerror0=(x0.T@x0-target.T).square().sum()\noffset0=torch.tensor(inp.W,device=\'cpu\')\nh1=inp.ref[\'oei\']-.5*np.einsum(\'prrq->pq\',inp.ref[\'tei\'])\nconstant=torch.tensor(h1-.5*inp.ref[\'electrons\']*inp.S+inp.beta_1*np.eye(76),device=\'cpu\')\nenergy=float(inp.energy_upper-inp.ref[\'ecore\']+inp.ref[\'electrons\']*inp.beta_1)\none=torch.tensor(uo,device=\'cpu\')\nhp0=constant-torch.einsum(\'rb,rbp,rbq->pq\',(w0*offset0[:,None]).sum(-1),ut,ut)\n_,eigvec=torch.linalg.eigh(hp0)\nrows,cols=linear_sum_assignment(-np.abs(uo@eigvec.cpu().numpy()))\npermutation=np.empty(76,dtype=np.int64);permutation[rows]=cols\nperm=torch.tensor(permutation,device=\'cpu\')\ncut=r*b*c;size=cut+r*c;calls=0;best=float(base[\'lambda_eff\']);best_z=np.zeros(size);history=[]\ne_cap=.01999995;gap_floor=3.9149\n\ndef quantities(flat):\n    dv=flat[:cut].reshape(r,b,c);off=offset0+flat[cut:].reshape(r,c)\n    vv=v0+dv;w=torch.linalg.solve_triangular(Rt,vv,upper=True)\n    delta=v0@dv.transpose(1,2)+dv@v0.transpose(1,2)+dv@dv.transpose(1,2)\n    dm=torch.einsum(\'rsij,sjk,rslk->ril\',cross,delta,cross)\n    e2sq=error0+2*(E0*delta).sum()+(delta*dm).sum()\n    if a.fixed_gap:\n        # Solve the scalar physical gap equation outside autograd, then attach\n        # its exact implicit first derivative. No eigenvector or shift changes.\n        C=torch.einsum(\'rb,rbp,rbq->pq\',(w*off[:,None]).sum(-1),ut,ut)\n        with torch.no_grad():\n            alpha=torch.ones((),device=\'cpu\')\n            offnorm=off.square().sum()\n            for _ in range(8):\n                ee,qq=torch.linalg.eigh(constant-alpha*C)\n                f=energy+ee.abs().sum()-ee.sum()+.5*alpha.square()*offnorm-gap_floor\n                projected=torch.diagonal(qq.T@C@qq)\n                derivative=alpha*offnorm-((ee.sign()-1)*projected).sum()\n                alpha-=f/derivative\n                if abs(float(f))<2e-12:break\n        ee=torch.linalg.eigvalsh(constant-alpha*C)\n        f=energy+ee.abs().sum()-ee.sum()+.5*alpha.square()*off.square().sum()-gap_floor\n        implicit_alpha=alpha-f/derivative\n        off=off*implicit_alpha\n    hp=constant-torch.einsum(\'rb,rbp,rbq->pq\',(w*off[:,None]).sum(-1),ut,ut)\n    ev=torch.linalg.eigvalsh(hp)\n    hq=torch.einsum(\'j,jp,jq->pq\',ev[perm],one,one)\n    e1sq=(hq-hp).square().sum()\n    lam1=ev.abs().sum()\n    lam=lam1+.25*(w.abs().sum(1)+(off+w.sum(1)).abs()).square().sum()\n    gap=energy+lam1-ev.sum()+.5*off.square().sum()\n    effective=(gap*(2*lam-gap)).sqrt()\n    return effective,e1sq,e2sq,gap,lam,w,off\n\n\n# V6 addition: a small constrained SQP over 8 rank scales and 480 offset coordinates.\n# The 68,640 whitened weight coordinates are restricted to rank scaling here.\ncurrent=np.zeros(r+r*c);best_native=float(base[\'lambda_eff\']);best_coords=current.copy()\ncache_x=None;cache=None;evals=0\n\ndef evaluate_sqp(x,eps):\n    global cache_x,cache,evals,best_native,best_coords\n    key=(eps,x.tobytes())\n    if cache_x==key:return cache\n    xx=torch.tensor(x,device=\'cpu\',requires_grad=True)\n    rank=torch.exp(xx[:r]);dv=(rank[:,None,None]-1)*v0\n    off_delta=xx[r:].reshape(r,c)\n    flat=torch.cat([dv.reshape(-1),off_delta.reshape(-1)])\n    value,e1,e2,gap,Lam,w,off=quantities(flat)\n    sm=.25*(torch.sqrt(w.square()+eps*eps).sum(1)+torch.sqrt((off+w.sum(1)).square()+eps*eps)).square().sum()\n    exact=.25*(w.abs().sum(1)+(off+w.sum(1)).abs()).square().sum()\n    loss=.5*torch.log(gap*(2*(Lam+sm-exact)-gap))\n    cs=torch.stack([1-e1/e_cap**2,1-e2/e_cap**2])\n    grad=torch.autograd.grad(loss,xx,retain_graph=True)[0].detach().numpy()\n    jac=np.stack([torch.autograd.grad(cs[i],xx,retain_graph=True)[0].detach().numpy() for i in range(2)])\n    val=float(value.detach());cv=cs.detach().numpy()\n    if cv.min()>0 and val<best_native:\n        best_native=val;best_coords=x.copy()\n    cache=(float(loss.detach()),grad,cv,jac)\n    cache_x=key;evals+=1\n    if evals%20==0:print(json.dumps(dict(evals=evals,lambda_eff=val,best=best_native,error_1=float(e1.detach().sqrt()),error_2=float(e2.detach().sqrt()))),flush=True)\n    return cache\n\nfor eps in [1e-7,1e-9,1e-11]:\n    result=minimize(lambda x:evaluate_sqp(x,eps)[:2],current,jac=True,method=\'SLSQP\',\n        constraints=[dict(type=\'ineq\',fun=lambda x:evaluate_sqp(x,eps)[2],jac=lambda x:evaluate_sqp(x,eps)[3])],\n        bounds=[(-.01,.01)]*r+[(-.03,.03)]*(r*c),options=dict(maxiter=150,ftol=1e-12,disp=False))\n    print(\'SQP\',eps,result.success,result.message,flush=True)\n    current=best_coords.copy()\nwith torch.no_grad():\n    xx=torch.tensor(best_coords,device=\'cpu\');dv=(torch.exp(xx[:r])[:,None,None]-1)*v0\n    flat=torch.cat([dv.reshape(-1),xx[r:].reshape(-1)])\n    got=quantities(flat);w,off=got[-2:]\n    result=dict(z);result[\'W\']=w.numpy();result[\'H2_I\']=(off+w.sum(1)).numpy();result[\'offset_W\']=off.numpy()\nnp.savez(a.out/\'submission.npz\',**result)\nnative=V.evaluate_words(V.load_inputs(a.out/\'submission.npz\'),result[\'angle_words\'],result[\'tree_edges\'],int(result[\'brot\']))\nassert V.passes_accuracy(native)\nreport=dict(method=\'rank scales + offsets SQP\',before={k:float(base[k]) for k in [\'lambda_eff\',\'error_1\',\'error_2\',\'E_gap\']},after={k:float(native[k]) for k in [\'lambda_eff\',\'error_1\',\'error_2\',\'E_gap\']},evals=evals,seconds=time.time()-start)\n(a.out/\'report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)\n', 'research_cpu/q8lib.py': '"""GPU helpers for 8-bit DF-THC research: target, Givens tree, word decoding, two-body error.\n\nThe two-body error is the official error_2 (Frobenius norm of the shifted two-body residual),\nevaluated in the orthonormal symmetric pair basis (2926 coordinates for N = 76).\n"""\nfrom __future__ import annotations\n\nimport math\nimport os\nimport pathlib\nimport sys\n\nimport numpy as np\nimport torch\n\nN = 76\nCOMP = pathlib.Path(os.environ[\'QCC_COMP_DIR\'])\nTARGET_CACHE = pathlib.Path(os.environ.get(\'QCC_TARGET_CACHE\', \'outputs/search/target_ts64.npy\')).resolve()\nDEV = \'cpu\'\n\n\ndef build_target64() -> np.ndarray:\n    if TARGET_CACHE.exists():\n        return np.load(TARGET_CACHE)\n    sys.path.insert(0, str(COMP / \'verifiers\'))\n    from dfthcsos.scoreinfra.dfthc_math import symmetric_from_lower\n    from dfthcsos.scoreinfra.reference import load_fixed_shifts, load_reference\n    base = COMP / \'verifiers\' / \'dfthcsos\'\n    ref = load_reference(base / \'FeMoCo76o113e.fcidump\')\n    shifts = load_fixed_shifts(base / \'FeMoCo76o113e.fixed_shifts.npz\')\n    s = np.asarray(symmetric_from_lower(shifts[\'Hsym_lower\'], N), dtype=np.float64)\n    t = np.asarray(ref[\'tei\'], dtype=np.float64).copy()\n    d = np.arange(N)\n    t[:, :, d, d] += 0.5 * s[:, :, None]\n    t[d, d, :, :] += 0.5 * s[None, :, :]\n    rows, cols = np.tril_indices(N)\n    scale = np.where(rows == cols, 1.0, math.sqrt(2.0))\n    pairs = rows * N + cols\n    ts = t.reshape(N * N, N * N)[np.ix_(pairs, pairs)] * np.outer(scale, scale)\n    np.save(TARGET_CACHE, ts)\n    return ts\n\n\nclass Pair:\n    """Map unit vectors u (..., N) to symmetric pair coordinates of u u^T (..., 2926)."""\n\n    def __init__(self, dev=DEV, dtype=torch.float64):\n        rows, cols = np.tril_indices(N)\n        self.ri = torch.tensor(rows, device=dev)\n        self.ci = torch.tensor(cols, device=dev)\n        self.scale = torch.tensor(np.where(rows == cols, 1.0, math.sqrt(2.0)), device=dev, dtype=dtype)\n\n    def __call__(self, u):\n        return u[..., self.ri] * u[..., self.ci] * self.scale\n\n\nclass Tree:\n    """Level-parallel Givens tree (official parent-before-child order): angles (units of pi) -> unit vectors."""\n\n    def __init__(self, edges, dev=DEV, dtype=torch.float64):\n        edges = [(int(p), int(q)) for p, q in edges]\n        self.edges = edges\n        levels, cur, used = [], [], set()\n        for t, (p, q) in enumerate(edges):\n            if p in used or q in used:\n                levels.append(cur)\n                cur, used = [], set()\n            cur.append(t)\n            used.update((p, q))\n        levels.append(cur)\n        self.plan = []\n        for level in levels:\n            colmap = np.full(N, N - 1, dtype=np.int64)\n            sgn = np.zeros(N)\n            perm = np.arange(N)\n            for t in level:\n                p, q = edges[t]\n                colmap[p] = colmap[q] = t\n                sgn[p], sgn[q] = -1.0, 1.0\n                perm[p], perm[q] = q, p\n            self.plan.append((torch.tensor(colmap, device=dev), torch.tensor(sgn, device=dev, dtype=dtype),\n                              torch.tensor(perm, device=dev)))\n        self.dev, self.dtype = dev, dtype\n\n    def __call__(self, theta):\n        """theta: (..., N-1) in units of pi."""\n        shp = theta.shape[:-1]\n        th = theta.reshape(-1, N - 1)\n        k = th.shape[0]\n        ang = math.pi * th\n        one = torch.ones(k, 1, device=th.device, dtype=th.dtype)\n        c = torch.cat([torch.cos(ang), one], dim=1)\n        s = torch.cat([torch.sin(ang), 0 * one], dim=1)\n        u = torch.zeros(k, N, device=th.device, dtype=th.dtype)\n        u[:, 0] = 1.0\n        for colmap, sgn, perm in self.plan:\n            u = u * c[:, colmap] + u[:, perm] * (s[:, colmap] * sgn.to(th.dtype))\n        return u.reshape(*shp, N)\n\n\ndef signed_words(words, brot):\n    half = 1 << (brot - 1)\n    return ((np.asarray(words, dtype=np.int64) + half) % (1 << brot)) - half\n\n\ndef canonical_gauge_words(words, edges, brot):\n    """Exactly preserve vector projectors while moving Givens words to the sign gauge.\n\n    Integer sign propagation, after marculera\'s onebody.signed_gauge (Apache-2.0,\n    kaggle.com/datasets/marculera/dfthc-merge-pipeline). Independent clipping or\n    reduction modulo pi at every edge would change a subtree\'s relative signs.\n    """\n    shape = np.asarray(words).shape\n    edges = np.asarray(edges, dtype=np.int64)\n    out = np.asarray(words, dtype=np.int64).reshape(-1, len(edges)).copy() % (1 << brot)\n    mod, half, quarter = 1 << brot, 1 << (brot - 1), 1 << (brot - 2)\n    signs = np.ones((len(out), len(edges) + 1), dtype=np.int8)\n    for k in reversed(range(len(edges))):\n        p, q = edges[k]\n        angle = np.where(signs[:, p] != signs[:, q], -out[:, k], out[:, k]) % mod\n        flip = (angle >= quarter) & (angle < 3 * quarter)\n        out[:, k] = np.where(flip, (angle + half) % mod, angle)\n        signs[flip, p] *= -1\n        signs[:, q] = 1\n    return out.reshape(shape)\n\n\ndef theta_of_words(words, brot):\n    """Official convention: theta / pi = word / 2**(brot - 1) (mod 2)."""\n    return signed_words(words, brot).astype(np.float64) / (1 << (brot - 1))\n\n\ndef gauge_angles(u, edges):\n    """Continuous angles (units of pi) of unit vectors in the signed half-open gauge, reproducing +-u."""\n    a = np.array(u, dtype=np.float64, copy=True)\n    ang = np.zeros((a.shape[0], len(edges)))\n    for j in reversed(range(len(edges))):\n        p, q = edges[j]\n        norm = np.hypot(a[:, p], a[:, q])\n        sign = np.where(np.abs(a[:, p]) < 1e-12, np.where(a[:, q] > 0, -1.0, 1.0), np.where(a[:, p] < 0, -1.0, 1.0))\n        ang[:, j] = np.where(norm < 1e-12, 0.0, np.arctan2(a[:, q] * sign, a[:, p] * sign))\n        a[:, p] = np.where(norm < 1e-12, 0.0, norm * sign)\n        a[:, q] = 0.0\n    return ang / np.pi\n\n\nclass TwoBody:\n    """error_2 = || sum_{r,c} x_rc x_rc^T - T ||_F with x_rc = sum_b w_rbc p(u_rb)."""\n\n    def __init__(self, dev=DEV, dtype=torch.float64):\n        self.T = torch.tensor(build_target64(), device=dev, dtype=dtype)\n        self.pair = Pair(dev, dtype)\n        self.dev, self.dtype = dev, dtype\n\n    def X(self, U, w):\n        v = self.pair(U)                                  # (R, B, P)\n        return torch.einsum(\'rbm,rbc->rcm\', v, w).reshape(-1, v.shape[-1])   # (R*C, P)\n\n    def resid(self, U, w):\n        a = self.X(U, w)\n        return a.T @ a - self.T\n\n    def err(self, U, w):\n        e = self.resid(U, w)\n        return torch.sqrt((e * e).sum())\n\n\ndef load_submission(path):\n    z = np.load(path)\n    out = {k: np.array(z[k]) for k in z.files}\n    return out\n\n\ndef dpen(w):\n    a = w.abs().sum(dim=1)\n    s = w.sum(dim=1)\n    return 0.5 * (a * a - s * s).sum()\n', 'verify_reachable.py': '"""Exact host-domain ANGLE proof, all addresses and independent HMR outcomes.\n\nSPDX-License-Identifier: Apache-2.0\nAdapted from mrmeows\' Apache-2.0 audit_affine_hmr.py. Ahmed Berat Ozer:\nreachable-domain initialization, both traversals, hash-bound certificates.\nThe address is fixed symbolically across a bit-packed vector of all valid\naddresses; each measurement contributes an independent linear phase mask.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport kickmix as km\n\nHOST = \'https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092\'\n\n\ndef packed(bits):\n    return int.from_bytes(np.packbits(np.asarray(bits, np.uint8), bitorder=\'little\').tobytes(), \'little\')\n\n\ndef prove(text, initial, expected, nq):\n    count, words = initial.shape\n    assert expected.shape == initial.shape and words == 5\n    n = (count - 1).bit_length()\n    full = (1 << count) - 1\n    values = [0] * nq\n    for i in range(n): values[i] = packed((np.arange(count) >> i) & 1)\n    addresses = values[:n].copy()\n    for j in range(words):\n        for bit in range(8): values[n + 8*j + bit] = packed((initial[:, j] >> bit) & 1)\n    phase = []\n    classical = {}\n    ccx = 0\n    for line in text.splitlines():\n        fields = line.split()\n        if not fields: continue\n        name = fields[0]\n        if name in (\'REGISTER\', \'APPEND_TO_REGISTER\'): continue\n        condition = None\n        if \'if\' in fields:\n            index = fields.index(\'if\')\n            assert len(fields) == index + 2\n            condition = fields[index+1]\n            fields = fields[:index]\n        ids = [int(q[1:]) for q in fields[1:] if q.startswith(\'q\')]\n        if name not in (\'CZ\', \'Z\', \'NEG\'):\n            assert condition is None, (\'unsupported conditioned value gate\', line)\n        if name == \'X\':\n            q, = ids; values[q] ^= full\n        elif name == \'CX\':\n            a, b = ids; values[b] ^= values[a]\n        elif name == \'CCX\':\n            a, b, c = ids; values[c] ^= values[a] & values[b]; ccx += 1\n        elif name == \'HMR\':\n            q, = ids\n            bit = next(x for x in fields[1:] if x.startswith(\'b\'))\n            classical[bit] = len(phase)\n            phase.append(values[q])\n            values[q] = 0\n        elif name == \'R\':\n            q, = ids; assert values[q] == 0, (\'discarded information\', line)\n        elif name in (\'CZ\', \'Z\', \'NEG\'):\n            assert condition in classical, (\'unsupported phase condition\', line)\n            correction = full\n            for q in ids: correction &= values[q]\n            phase[classical[condition]] ^= correction\n        else:\n            raise AssertionError((\'unsupported instruction\', line))\n    assert values[:n] == addresses, \'selection changed\'\n    for j in range(words):\n        for bit in range(8):\n            assert values[n+8*j+bit] == packed((expected[:, j] >> bit) & 1), (\'wrong reachable word\', j, bit)\n    assert not any(values[n+8*words:]), \'dirty scratch\'\n    assert not any(phase), \'measurement phase is not paid for every outcome\'\n    return dict(status=\'PASS\', all_addresses=count, independent_HMR_outcomes=len(phase),\n                all_measurement_assignments=f\'2^{len(phase)}\', zero_phase_every_outcome=True,\n                selection_restored=True, scratch_zero=True, CCX=ccx, Q=nq)\n\n\ndef verify(folder, out):\n    from dfthcsos.scoreinfra.circuits import register_layout\n    started = time.time()\n    with np.load(folder/\'submission.npz\', allow_pickle=False) as z:\n        words = np.asarray(z[\'angle_words\'], dtype=np.int64)\n    assert words.shape == (1220, 75)\n    batches = words.reshape(len(words), 15, 5)\n    report = dict(status=\'RUNNING\', contract=\'official reachable forward/reverse states\',\n                  host_clarification=HOST, arbitrary_target_claim=False, circuits=[])\n    for k in range(15):\n        file = folder/f\'angle_k_{k}.kmb\'\n        c = km.Circuit.from_file(file, format=\'kmb\')\n        layout = register_layout([c])\n        live = [q.id for q in layout.selection] + [q.id for w in layout.words for q in w]\n        assert live == list(range(51)), \'unexpected declared register layout\'\n        previous = np.zeros_like(batches[:, k]) if k == 0 else batches[:, k-1]\n        current = batches[:, k]\n        forward = prove(str(c), previous, current, c.num_qubits)\n        reverse = prove(str(c), current, previous, c.num_qubits)\n        row = dict(batch=k, sha256=hashlib.sha256(file.read_bytes()).hexdigest(),\n                   forward=forward, reverse=reverse)\n        report[\'circuits\'].append(row)\n        out.write_text(json.dumps(report, indent=2))\n        print(json.dumps(dict(batch=k, forward=\'PASS\', reverse=\'PASS\', Q=c.num_qubits,\n                             T=c.operation_statistics().num_toffoli_like)), flush=True)\n    report.update(status=\'PASS\', seconds=time.time()-started)\n    out.write_text(json.dumps(report, indent=2))\n    return report\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(); p.add_argument(\'folder\', type=Path); p.add_argument(\'out\', type=Path)\n    a = p.parse_args(); result = verify(a.folder, a.out)\n    print(json.dumps(dict(status=result[\'status\'], seconds=result[\'seconds\'])), flush=True)\n', 'LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'upstream/heliosli_v26.ipynb.zlib.b64': 'eNq1Wmty28gRvsqE+wOgCwD1tkSvfsiWvKusHWtFZb0VScUMgSExJl6aAShRsqpyhPxI5RY5RW6yJ8nXA4AEH5a8rqzKtgiwu6cfXz9mxg8tX0SRbnUvH8ynfj7NRKvbirkaB+lt0nJasch5wHPe6j48Oi2dFsoniu/YRSgYfaFFzqRmWTGIpM+S9NZjJ8mndPqn3/7zz/LPVXKVvE0VE9wPGRjGLA/BkaS5GKQpnvhYaLwTzA8Fz4TOmS+VX8ic3Qo2hkiWDh2mhDsRSg4liAU+TdlQRoKFQgl2K/OQJFwlRRKnAREFYBpKX/KIVWxKO4wnAbtVMoeMv3fGfDSKROc2VWOZjDq6GMRSa5km3r3M/u6R3p/ZBSn8mb2pNPrMznkuU/YZ37muW//t0jOTyUTfKCI6CmNo8FoonrP//vteKEuzvS33phhAhtHbHqo0ZtN0mg44vm2DaWvD2325Y0Sz459P8SYUycgfb22Dm2eZSu9kzHPB9vcPKlGB8NNAKJDuHni7e/slczDMQ1+nmhSBd3nCjqJAxJK0iARXCXTjCWxnsD3QLE1YL43Fn1OZ5CA5Z4ds32Gv8WtzZ9thb/Bhb4MNuZ+nSt6T/YmDsA1IgJCjMIeIQunKpuNY5kpK9kNU3AfpxILbe1yNEPKfQkCA52rKE3q7xkkO03IU80GqiNGEa8BlwCOZ8EBG9yFxtp1K+womoKM/WsTQCAxTmCvu2JtQRjlMs4dyIoAGAkhsFCVnH3gHW3BX69H53dj/jp1XWJxWSPTTOEsTkeQGjgScEwK7CTTs9tOMAGm8QzhXAlHQ8/whMxFYwD9gecrIot6PR+7W7t4rok8MUx4qIdaAei6WtBC5pPAY0UwVCcVW5g6b5wWclxO5VEDPkBdRznSYIufSIsm1t+IQAhicIe6EX5DoviFsdZMiipZ9lBZ5VuRUUK4bDpNwDrJiFKUDh4Vch5HEh0+aUBTdx9wh40OHpRT+EItE+D3FQy5j4TAkIznSYRmowEruhebwbhL0yWa73b1KGH6GqDIZspBdptoTyUQq5PJI5Lb185s3/Tcf3p/1j0/PLYdZFjBk1QVAJlC60/Ce7ozSFN9UBYeAjnzjSV7EboU5l8hlZHLBcsrl5z9Lon+ftOvuXJwcwiCCBwwi8z2pA6ns+ukTMtbOYMoMDVa73V3URom8UAnLyreKSwCvN9W5iE/uJFyzghoUZniySAKrveDqCrUzb0O3dW4+7hknN9WoVJhTXzYpr0vCkGB/aFDi0T/2khNfvOgQ0/nJu5Oj3kn//dFfTt+e9C48wpFF7cFHZiPTDy9UIdozFckaEt1UZ8UHywm54IJFC4zfEYOEx8ImwZcb18ZPBC+HHfdgQwOZzpLvrpKPH85/Ak0FZu8Mv+11fiS6Gq5LrQpBvkp6f6UCbaR1mDVvXtZVguQxel52N65Bc7kAl1JPayz9cSzvKBHWfd0AFOJTZXDFQ+V2HF8lpvDUPa8iKctif65OvyYA0wSfK7bgRn6Zhb4kcvyuyeuOtsBTEg3zq6RGA6wlPHhRygM7zUSymCrHPZj2ZRS1ya8nv56dvLk4Oe6ffTi/ePvh3ekHCLUmO7v99B6+pT4DBWrGS4v0GaaRTC34+pCtsjvMtnTOozm8oL2JlLNejLNGCAX86B3ZN2fRPMopezKFvr2ayF3Ir6L5ebnl0HfGGZ8ZSaFHEt/+A6o/1Q8dcjQzmwJR14UQplS9wKu+rnLNDHMmdplpC5YaWG0K9bCRw1Tq/bBIxlTuMc8pO+LxIOBdNvSUQPA32fffs60twHtgWcs1MfSKDHoL24hYzPHQC8VdIEeYQ22T2CgiyDVP3Emd61n1KxuVp+IcbdkGAWhJKehCSdEIEzUvPavqWvkOCzDjHrIVaBIvwoopSwR9Dh5oT2neqb4hSXW5hAhwK8wcXjymnlA+aFP+HGa07afjpWq4sgLqTqDJ47bl3d0vOMrEgfqzZ4IBxcsoSKeMDjRAcG7r4KRLPq4chNlnSnqng082sQ5Th9WxKRlEpEWDd4nPrh1WUY9Scl2FmPnrKi3Nt4e1mYaI8slu+s8hqnadNhEsWYkVTQfmYz26LU1oNG3FHMigzQrmzWpas/6I/Dk/ebuQ9UoMMWMmvoAoAJLwoITGGEfd8+HxKsk38IFGJ4/+IQzTlIF349irtjEeFdV+6d5c2SXGrKpSe+N4QG0BaIbJh5Z5nC1yWZNRuTMV3SuLsU2rOOWbM5Viyol7mfBPk0mPhEJaP+ln9uYGRFPNOTQFZ1a96jYRYI6mdk7d125YwVyWE6uFjoSp+FttREt5zj6QlLYR7aJt9KZhGzZp3gChCGnTgEaHKo+42EsGBiIyO5WZqdTevtXMuZZVR6xUHea1qk1ja5r1Lp811a9RBkunt5jJFQ2pBDWAFH/Lomd29Sh6szDS0EJ+dJpKVEmuwFybQYzXCxmsqDzxvNBlO7XOjno9iLGJEqvXtdpswg8ZkqOUAXOHaPq5Tfw+ypt13STVHvbPKHW2eZrXQ6M4LVPr3ShE4i5XnKa5q9YDpCZF3Debbmj2yMpPDqNvcmzK0Ln74qaQE/R5bKKvu97m8JFdlN+482/0Vauues3lyVdrlra/em1TVmjZDSwbC2z5q7VBVy+58GMkl9MmNCtEXyiVYtxz2Eb7kZlX8GlNkqFVL5GYV6x8ddVaX8sbLizbc18Mh0bv+SMpvT18fMKcuSVf8kVtIUG0jnSFmBVUVBB2SuXaleJlSpBTiK17oB8ZIY9RjWUPK0K63t7osYbhg/nV3fd2oCyzH4zgx7ZxyiiG+bS99TKVVvDTbfbiBcOE0mHbs2wcWiORoh8o6ZfxS4dsVupBSYp02cMo7tIyazrNVx1cfKSzL9O+5uP2H9CzsGXv02izdoNCp2sYpM14Ue3tvb/J7C1V6pqRBgvgjEZalAriOpyRnmIXf/L23RHmYzN53Dd2/89NX/Rz75kTQHt1snIWnhCadUNeVTxvVZpT0ZxrXJbP+tmjKma38auv5T1FcOvFi60NDD9UUN/L198WQtYzH8ujMR9THKEfQ6oLSFU1DNj67R//ev4IEpPxFGAgt9EHZ36QVO1P2OYuK+MLMJL/wjzPdLfTub299codKca0uEN46XBabkCrpVirUx9wcOkGQxcdwB3BP5nbiCiz6RjYD3mCWZsdZRzTlbvloeHQWoNCRuacSgk6w5odlC4foz6lU8XUiQbu1pZ7cOBWHnI3NzbdWo42yzWWN64lN0QyEXwkKjeLoNSmy47GMY/Yr2mAiXvC7EGhpxyoiAUiG6tYmD55FhY+uwi5ZPaxOJOitGrpbJROvhS4tVfHEI2git8Xzn7rE18aaU01SdjG5k57bTCfiFm1D9SdepnZBxcquLsH7u5LZs+98mp2Xv8cEGYCSQ5mK5fnRtze/i6piYSFUDDBn0jUn6SSUcQuprEPNIQygbmzrfBzS40Ncz6NO0vnadoVd9zP3REvRsLddemY272BRzx2XPmvDi70oaOcOri0ZPMsHNOUMF8HDb6nUafBTlu6fFUroPDlSzdKfR65o9idneOTz9uvltdGngiu6NIEKhhU/nB24e6xI01dFXPoq9Ukf7Vw9g0h+/vbVb70sNlVtR3AC+IfILVo1U8Bnx7sbW/ugOHnUumj0znUamtHKNnFwFg64+jM6DGvztC/ole1gzJVn/s8p1XtmMYfQPuT8AnIELCSH0ejRE7Ya54I9UnA8TP/AEq/SNSbC/hmTbZ5VTpVw2ddE7/hNsReVAFvKBztL9ySkF0LFyWeuSkbymEuRLJ0b7GStAv3GDJZjuUsMZj99XUZIvxSQgfw2wX8BvVFWuCaSwWXRh53JtxhzbTPhHJ1lOa1YtNZOaSVy/qMHXEjtO0uIwbNthy2beK+ucPW3ww9p3xQMoxK+s4Tp+YuLeCWfdKp1qdm67Adh+0aLfYbl1S0MJ4Ywh9JoLPqeHS1VF7QYOozKi8j5ll3G3pekndufB9Jvn/jKpFxqVyo4SbQdyJcOBjzUK6msMINhJYjNMTF4vj/L39tj71L+TPlb9FkB40IoNCRdFazcylVSjNeLSZAvVSXvZdo9tgMnxeYjYKFIkHrmjrhPNldf4l4IHXEJ+wdn9BxF1FEdAOZyGQ02difd+Ay7xYvKe2PDvtxq3/q0DUaul//I5o9zRcF3Q1DKZVDzBewyux6LNo++Pqx6FsR3ByHzEkcLVSjla4BltSlUaoGMdt+aQC/vV/NdOuuXwG2Cb/n4+LVfNyrzDLJMXwa5iVvZ0jlY8QzV99ki+2saYKzPM01Uo8WqxQoJ68y7dYMr/YEdYQoJrO5VP8hg6mzMg0Wmhxt3F5Xj4V76hk0GgPzk0rNuako7+/u7h1QXW6mbdn7F3ShdddfiT89kSzT106ADySsTifoBOV/4HBTP986WOgAtChdzZEkdot6UeiyOcWc7qLKC3W0LkxxMPo2RJLTxpXGpLKSWLpqxh/UiCfYC6m6HZvjj4Boy2tXp3GrTXbW9032D+YCtR5NGGaTDj1EmIMSZ+Xeu+21Hq8XtlCtsUCbj3QmfHpCDckiPu1T5cOm6myah5TW2HZFaNAFihXeZuYt3lVU5fM2Nm0zqr5MhikJXCBpPYIkGZTHi63uzvyhH8skVa3u7uP/AJHsGpY=', 'upstream/heliosli_v26_CREDITS.md': "# Sources and credits\n\n- **invsqrt** — Ahmed Berat Özer's 62-qubit file, byte for byte, from the version 15 output of\n  https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression (his changes Apache-2.0),\n  built there on yoyobar's 62-qubit file (https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits,\n  Apache-2.0) and the lineage credited there: Akmal Xodarev (busyaprime), mrmeows, Phuc Thai (DePie),\n  Dmitrii Gluzdov, marsyas.\n\n- **dqi** — hengck23's approximate decoder (iteration 014), byte for byte, from\n  https://www.kaggle.com/datasets/hengck23/hengck23-dqi-59-57 (Apache-2.0; notebook\n  https://www.kaggle.com/code/hengck23/dqi-kmb-at-59-56854), as packaged in Kirill Tymchishin's portfolio\n  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom. Decoder lineage as named there:\n  SomeJoint's streamed decoder line (https://www.kaggle.com/code/somejoint/quantum-circuits-22-77-local-gm-889-qubit-dqi);\n  SomeJoint's research team and GPT-6 Astra Pro; Ahmed Berat Özer; Alex Chilton's 883-qubit Schur decoder, based on\n  jday96314's QuantumAI decoder (https://github.com/jday96314/QuantumAI); mrmeows, Ahmed Berat Özer and manwithacat\n  (mask projection); Dmitrii Gluzdov, Agniv Banerjee (SomeJoint), ViberTeam, Phuc Thai (DePie).\n- **dfthcsos** — Atakan Aldemir's learned angle words on SomeJoint's (Agniv Banerjee's team) R = 8, B = 143, C = 60\n  factorization. The fifteen angle circuits, byte for byte, as assembled in Alex Chilton's portfolio (output of\n  https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio, Apache-2.0; per-slot assembly credited\n  there to manwithacat): slots 2, 3 and 14 from Dmitrii Gluzdov's\n  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, slots 0, 1, 4, 5 and 8 ours (from\n  our earlier versions), the rest from Atakan Aldemir's\n  https://www.kaggle.com/code/atakanaldemir/qcc-888q-repair-and-native-chemistry-co-design (as packaged in\n  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom). Loader lineage as named there:\n  Atakan Aldemir, heliosli, Dmitrii Gluzdov, on SomeJoint's design; factorization lineage: Michael Rustad, Ahmed Berat\n  Özer, Akmal Xodarev (busyaprime), Vladislav Lavrentev (lightningv08), mrmeows. The table weights (W, H2_I, offset_W) are ours, starting from Dmitrii Gluzdov's (version 39 output of\n  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source, Apache-2.0), with\n  earlier work starting from his versions 37 and 38, from Sargis Khachatryan's (avazaku; version 1 output\n  of https://www.kaggle.com/code/avazaku/free-gap-sqp-889-qubit-dqi, Apache-2.0, built there on our earlier\n  version) and from Ahmed Berat Özer's (v14 and v15 outputs of\n  https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression, Apache-2.0) and using work from\n  sigmaborov's (version 5 output of https://www.kaggle.com/code/sigmaborov/22-85569-lb-exact-gauge-search, Apache-2.0)\n  and baidalinadilzhan's (https://www.kaggle.com/code/baidalinadilzhan/quantumai-improved-public-oct29, Apache-2.0;\n  its code was used in making ours), all of which refine Atakan's.\n- **Organizers** — task definitions, verifiers and kickmix (Google Quantum AI / Qualtran, competition data)."}
SEARCH_RECORDS={'selection': {'baseline': {'path': 'data/top_v8/extracted/dfthcsos/submission.npz', 'valid': True, 'lambda_eff': 48.32336162007705, 'error_1': 0.019999410039273538, 'error_2': 0.019999441098101024, 'E_gap': 3.9148990000176127}, 'candidates': [{'path': 'data/top_v8/extracted/dfthcsos/submission.npz', 'valid': True, 'lambda_eff': 48.32336162007705, 'error_1': 0.019999410039273538, 'error_2': 0.019999441098101024, 'E_gap': 3.9148990000176127}, {'path': 'outputs/search/v8_sqp/submission.npz', 'valid': True, 'lambda_eff': 48.32304979921318, 'error_1': 0.019968382198724745, 'error_2': 0.019999868990438024, 'E_gap': 3.914900000047055}], 'selected_path': 'outputs/search/v8_selected_polished.npz', 'selected_lambda_eff': 48.32296860332425, 'polish': {'before': 287.22001993880303, 'after': 287.21901770894056, 'gram_defect': 1.5987211554602254e-14, 'seed': 20261030, 'radius': 0.1, 'history': [{'sweep': 0, 'objective': 287.2191853601439, 'accepted': 11337, 'seconds': 1.6572768688201904}, {'sweep': 1, 'objective': 287.219092232899, 'accepted': 10744, 'seconds': 3.3487491607666016}, {'sweep': 2, 'objective': 287.21905999288776, 'accepted': 10363, 'seconds': 4.9903669357299805}, {'sweep': 3, 'objective': 287.21904695346655, 'accepted': 10151, 'seconds': 6.637370824813843}, {'sweep': 4, 'objective': 287.21903904178754, 'accepted': 9892, 'seconds': 8.285216093063354}, {'sweep': 5, 'objective': 287.2190317689689, 'accepted': 9726, 'seconds': 9.924206972122192}, {'sweep': 6, 'objective': 287.219025320975, 'accepted': 9497, 'seconds': 11.58132004737854}, {'sweep': 7, 'objective': 287.2190216569668, 'accepted': 9337, 'seconds': 13.217747926712036}, {'sweep': 8, 'objective': 287.21901994734503, 'accepted': 9020, 'seconds': 14.852882862091064}, {'sweep': 9, 'objective': 287.2190190775515, 'accepted': 8910, 'seconds': 16.500802993774414}, {'sweep': 10, 'objective': 287.21901823245906, 'accepted': 8684, 'seconds': 18.14573097229004}, {'sweep': 11, 'objective': 287.21901770894056, 'accepted': 8590, 'seconds': 19.790275812149048}]}}, 'v8_sqp': {'method': 'rank scales + offsets SQP', 'before': {'lambda_eff': 48.32336162007705, 'error_1': 0.019999410039273538, 'error_2': 0.019999441098101024, 'E_gap': 3.9148990000176127}, 'after': {'lambda_eff': 48.32304979921318, 'error_1': 0.019968382198724745, 'error_2': 0.019999868990438024, 'E_gap': 3.914900000047055}, 'evals': 3048, 'seconds': 65.75599479675293}}
for relative,text in SOURCES.items():
    path=SRC/relative;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text)
(SRC/'upstream/heliosli_v26.ipynb').write_bytes(zlib.decompress(base64.b64decode(SOURCES['upstream/heliosli_v26.ipynb.zlib.b64'])))
(WORK/'search_records_v8.json').write_text(json.dumps(SEARCH_RECORDS,indent=2))
print('Search code, records, original V26 notebook and upstream credits saved.')


## Selected coefficients and exact pair polishing

In [ ]:
WEIGHT_BYTES='...'
WEIGHT_SHA256='b79f3cec4c85c79ccee990d9ecf595ab61df86bddb85cc6ddb4dc342b8dbcb70'
raw=zlib.decompress(base64.b64decode(WEIGHT_BYTES));assert hashlib.sha256(raw).hexdigest()==WEIGHT_SHA256
factor=SUB/'dfthcsos/submission.npz'
with np.load(factor,allow_pickle=False) as z:base={k:np.array(z[k]) for k in z.files}
with np.load(io.BytesIO(raw),allow_pickle=False) as z:selected=dict(base,W=np.array(z['W']),H2_I=np.array(z['H2_I']))
if 'offset_W' in selected:selected['offset_W']=selected['H2_I']-selected['W'].sum(1)
checkpoint=WORK/'selected_checkpoint_v8.npz';np.savez(checkpoint,**selected)
from gauge_search import search
optimized=WORK/'production_optimized_v8.npz'
search_result=search(checkpoint,optimized,sweeps=8,radius=.1,seed=20261030)
with np.load(optimized,allow_pickle=False) as z:production={k:np.array(z[k]) for k in z.files}
for key in base:
    if key not in {'W','H2_I','offset_W'}:assert np.array_equal(base[key],production[key]),key
assert search_result['after']<=search_result['before']
numerical=chemistry.evaluate_words(chemistry.load_inputs(optimized),production['angle_words'],production['tree_edges'],int(production['brot']))
assert chemistry.passes_accuracy(numerical)
shutil.copyfile(optimized,factor)
print('Native deterministic accuracy checks pass; angle words and all immutable arrays unchanged.')


In [ ]:
RUN_ADDITIONAL_SEARCH=False
if RUN_ADDITIONAL_SEARCH:
    env=dict(os.environ,QCC_COMP_DIR=str(COMP),QCC_TARGET_CACHE=str(WORK/'target_ts64.npy'),PYTHONPATH=os.pathsep.join(native+[str(COMP/'verifiers'),str(SRC/'research_cpu')]))
    for script in ['fixed_word_joint64_v6.py','rank_offset_sqp_v6.py']:
        subprocess.run([sys.executable,str(SRC/'research_cpu'/script),str(checkpoint),str(WORK/script.removesuffix('.py')),'--fixed-gap'],env=env,check=True)
    subprocess.run([sys.executable,str(SRC/'gauge_multistart.py'),str(checkpoint),str(WORK/'additional_multistart'),'--rounds','3','--trials','6'],env=env,check=True)
    print('Optional research only; production remains the verified checkpoint above.')
env=dict(os.environ,PYTHONPATH=os.pathsep.join(native+[str(COMP/'verifiers')]))
proof=WORK/'reachable_proof_v8.json'
subprocess.run([sys.executable,str(SRC/'verify_reachable.py'),str(SUB/'dfthcsos'),str(proof)],env=env,check=True)
assert json.loads(proof.read_text())['status']=='PASS'


## Full receipts and fresh native verification

In [ ]:
INVERSE_FULL_RECEIPT={'status': 'PASS', 'score': 100940.60713369818, 'toffoli_equivalent': 1628.0743086080352, 'toffoli_count': 1331.4835533422884, 'max_toffoli_count': 1501, 't_count': 6.999725507403656, 'rotation_count': 41.0, 'rotation_eps': 2.434041480874084e-13, 'num_qubits': 62, 'reaction_depth': 805, 'num_ops': 47201, 'verified_shots': 200540928, 'worst_error_over_tolerance': 0.9993237694056958, 'worst_x': 161, 'circuit_sha256': '6dbb98b34c3b16f1c4e019350c10d42259a6faa97e464c9b7c9390f877f392f4'}
DQI_FULL_RECEIPT={'status': 'PASS', 'score': 408399264306.69824, 'm': 4095, 'n': 70, 'degree': 12, 'r': 2016, 'verified_shots': 102400, 'value_errors': 0, 'phase_errors': 0, 'empirical_error_rate': 0.0, 'effective_error_rate': 0.0008660587301657926, 'log2_delta': -128.0, 'repetition_factor': 1.1912604648115832, 'mean_error_weight': 31.046737331746655, 'toffoli_count': 433785.1698242187, 'max_toffoli_count': 434512, 'reaction_depth': 4337, 'num_ops': 60050827, 'shot_operation_counts': {'NEG': 3480030, 'BIT_INVERT': 164212903520, 'BIT_STORE0': 11210444800, 'BIT_STORE1': 6350298, 'X': 26432774784, 'Z': 3055106960, 'R': 6734252641, 'HMR': 9222163041, 'CX': 5468086053571, 'CZ': 40951277278, 'SWAP': 160603937767, 'CSWAP': 1327104000, 'CCX': 43009962990, 'CCZ': 82534400, 'INFO_TAG': 1681817600, 'POP_CONDITION': 49356800, 'PUSH_CONDITION': 49356800}, 'num_qubits': 889, 'num_bits': 101853, 'circuit_sha256': '7a378f4755605e0c5fa6117f808891bf8dcf0797d047155e6d4c9e75e3c99421', 'wall_seconds': 65.57770276069641}
RUN_FULL_UNCHANGED_VERIFIERS=False
if RUN_FULL_UNCHANGED_VERIFIERS:
    tasks={'invsqrt':[COMP/'verifiers/invsqrt/verify_submission_invsqrt.py',SUB/'invsqrt.kmb','--max-workers','4'],'dqi':[COMP/'verifiers/dqi/verify_submission_dqi.py',SUB/'dqi.kmb']}
    for task,cmd in tasks.items():
        run=subprocess.run([sys.executable,*map(str,cmd),'--salt','notebook-verification'],env=env,capture_output=True,text=True,check=True)
        receipt=json.loads(run.stdout[run.stdout.index('{'):])
        assert receipt['status']=='PASS'
        if task=='invsqrt':INVERSE_FULL_RECEIPT=receipt
        else:DQI_FULL_RECEIPT=receipt
assert sha(SUB/'invsqrt.kmb')==INVERSE_FULL_RECEIPT['circuit_sha256']
assert sha(SUB/'dqi.kmb')==DQI_FULL_RECEIPT['circuit_sha256']
assert INVERSE_FULL_RECEIPT['status']=='PASS' and INVERSE_FULL_RECEIPT['verified_shots']==200540928
assert DQI_FULL_RECEIPT['status']=='PASS' and DQI_FULL_RECEIPT['verified_shots']==102400
fresh=chemistry.verify(str(SUB/'dfthcsos'),salt='notebook-verification')
assert fresh['status']=='PASS' and fresh['verified_shots']==78080
assert fresh['score']<baseline_fresh['score'],'Must improve the frozen public leader'
REFERENCE={'invsqrt':2076774.233715282,'dqi':24439474177519.598,'dfthcsos':3355599588.3155365}
results=dict(invsqrt=INVERSE_FULL_RECEIPT,dqi=DQI_FULL_RECEIPT,dfthcsos=fresh)
ratios={t:REFERENCE[t]/r['score'] for t,r in results.items()}
local_score=math.prod(ratios.values())**(1/3)
projected_public_score=23.00037*(baseline_fresh['score']/fresh['score'])**(1/3)
for task,r in results.items():print(task,'PASS','cost',r['score'],'shots',r['verified_shots'])
display(Markdown(f'### Measured local score: **{local_score:.8f}**'))
display(Markdown(f"Chemistry cost: **{baseline_fresh['score']:.6f} → {fresh['score']:.6f}**. Projected Public Score: **{projected_public_score:.8f}** — official evaluation pending."))
print('DQI errors retained:',DQI_FULL_RECEIPT['value_errors'],DQI_FULL_RECEIPT['phase_errors'],'repetition penalty',DQI_FULL_RECEIPT['repetition_factor'])


In [ ]:
names=sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
assert names==sorted(PINS) and len(names)==18
hashes={name:sha(SUB/name) for name in names}
changed=[name for name in names if hashes[name]!=PINS[name]]
assert changed==['dfthcsos/submission.npz'],changed
ALLOWED={'X','CX','CCX','Z','CZ','CCZ','Z_POW','NEG','SWAP','CSWAP','R','HMR','BIT_INVERT','BIT_STORE0','BIT_STORE1','PUSH_CONDITION','POP_CONDITION','INFO_TAG','DEBUG_PRINT'}
for name in names:
    if name.endswith('.kmb'):
        c=km.Circuit.from_file(SUB/name,format='kmb')
        assert not(set(c.operation_statistics())-ALLOWED)
        assert max(c.num_qubits,c.num_bits)<2**30//1536
        assert all(isinstance(k,str) for k in c.register_data)
archive=WORK/'submission.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for name in names:z.write(SUB/name,name)
with zipfile.ZipFile(archive) as z:
    assert sorted(z.namelist())==names and z.testzip() is None
    for name,pin in hashes.items():
        with z.open(name) as f:assert hashlib.file_digest(f,'sha256').hexdigest()==pin
receipt=dict(status='PASS',local_score=local_score,results=results,reference=REFERENCE,ratios=ratios,
    upstream_kernel='heliosli/the-dataset-is-public-now-enjoy/26',upstream_public_score=23.00037,
    upstream_sha256=UPSTREAM_ZIP_SHA256,baseline_chemistry=baseline_fresh,projected_public_score=projected_public_score,
    changed_files=changed,files=hashes,archive_sha256=sha(archive),search=search_result,checkpoint_sha256=WEIGHT_SHA256,
    numerical={k:float(numerical[k]) for k in ['lambda_eff','error_1','error_2','E_gap']},fixed_salt='notebook-verification',
    leaderboard_score_measured=False,elapsed_seconds=time.monotonic()-started)
(WORK/'verification_receipt_v8.json').write_text(json.dumps(receipt,indent=2))
print('18-member verified submission ready:',sha(archive))


## Sources and credits

The **new baseline and its improvements** are [Ahmed Berat heliosli V26](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), official Public Score 23.00037. The original V26 notebook, component credits are included unchanged under `source_v8/upstream/`.

- **Inverse:** yoyobar's 62-qubit rework of busyaprime / mrmeows, with Özer's history cache removing one Toffoli. Earlier lineage: Phuc Thai / DePie, Dmitrii Gluzdov and marsyas. We retain these bytes unchanged and independently repeat the complete native verifier.
- **DQI:** hengck23's projective iteration 014 on SomeJoint / Agniv Banerjee's fused confluent Hermite line; selection by Kirill Tymchishin. Earlier lineage: alex chilton, jday96314, mrmeows, Ahmed Berat Özer, manwithacat, Dmitrii Gluzdov, ViberTeam and Phuc Thai. Approximation limitations and all measured error penalties remain intact.
- **Chemistry:** heliosli's newer coefficients selected by alex chilton, on loaders assembled per slot by manwithacat; circuits by Dmitrii Gluzdov, heliosli and Atakan Aldemir. SomeJoint's team supplies the R8/B143/C60 factorization. Coefficient lineage includes Sargis Khachatryan, sigmaborov, baidalinadilzhan, Ahmed Berat Özer, Michael Rustad, mrmeows, lightningv08, marculera and kozykappa. Base portfolio assembled by alex chilton V59.
- **Our V8 addition:** constrained coefficient continuation and rank/offset SQP on the fixed words, exact pair polishing, candidate selection, pinned baseline comparison and reproducible verification. The CPU fitter and q8lib derive from Özer's Apache-2.0 work; V8 feasibility caps retain a positive margin to the original 20 mHa limit. The gauge methods retain Kirill / mrmeows lineage.
- **Reachable proof:** Özer's adaptation of mrmeows' symbolic HMR audit. It covers both host-required traversals for every valid address and every independent HMR outcome; it makes no arbitrary-target claim.
- **Organizers:** Google Quantum AI / Qualtran, original benchmark data, Kickmix and unchanged verifiers. [Competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [benchmark paper](https://arxiv.org/abs/2610.23832), [measurement uncomputation](https://quantum-journal.org/papers/q-2018-06-18-74/).

Upstream components retain their authors' terms; our search additions are
Apache-2.0. The upstream compiled circuit designs are not our work.
